# Weather Data Pipeline Walkthrough

This notebook demonstrates the end-to-end weather data pipeline used in the assessment.

The pipeline:

1. Extracts daily historical weather data from the Open-Meteo Archive API.
2. Loads the extracted records into PostgreSQL.
3. Transforms the raw data using dbt.
4. Runs dbt data-quality tests.
5. Queries the resulting daily weather mart.
6. Demonstrates rerun safety / idempotency.

The Python extraction and loading functions used below are the same functions imported by the Airflow DAG.


In [ ]:
import sys
import subprocess
import os

import pandas as pd
import psycopg2

sys.path.insert(0, "/opt/airflow/ingestion")

from weather_api import fetch_daily
from load import load_data


def dbt(*args):
    result = subprocess.run(
        ["dbt", *args],
        cwd="/opt/airflow/dbt",
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)

    return result.returncode


def query(sql):
    conn = psycopg2.connect(os.getenv("DATABASE_URL"))

    try:
        return pd.read_sql_query(sql, conn)
    finally:
        conn.close()


print("Pipeline walkthrough environment ready.")


## 1. Extract

The extraction stage reuses the same `fetch_daily()` function used by the Airflow DAG.

The date is passed explicitly to the extraction function. This makes the extraction deterministic and allows historical dates to be processed during Airflow backfills.


In [3]:
logical_date = "2026-09-25"

records = fetch_daily(logical_date)

print(f"Extracted {len(records)} records for {logical_date}")

pd.DataFrame(records)


Extracted 5 records for 2026-09-25


,city,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum
0,Amsterdam,2026-09-25,14.414584,20.250000,9.60,0.0
1,Berlin,2026-09-25,13.062500,15.300000,10.95,1.2
2,Bengaluru,2026-09-25,22.731253,27.250000,21.00,6.7
3,New York,2026-09-25,16.945833,21.950001,12.35,1.5
4,Sydney,2026-09-25,19.608332,23.900000,15.60,0.0


## 2. Load

The extracted records are loaded into the raw PostgreSQL table.

The raw table uses `(city, date)` as its primary key. The loader uses `ON CONFLICT DO UPDATE`, making repeated loads for the same city/date idempotent.


In [4]:
load_data(records)

raw_count = query("""
    SELECT
        COUNT(*) AS rows_for_date,
        COUNT(DISTINCT city) AS cities
    FROM raw.weather_daily
    WHERE date = '2026-09-25'
""")

raw_count


Connecting to DB and inserting 5 records...
Load complete!


/tmp/ipykernel_34/165350804.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,rows_for_date,cities
0,5,5


### Raw data evidence

The raw layer preserves the weather fields returned by the extraction stage and stores one row per city/date.


In [5]:
raw_sample = query("""
    SELECT
        city,
        date,
        temperature_2m_mean,
        temperature_2m_max,
        temperature_2m_min,
        precipitation_sum
    FROM raw.weather_daily
    ORDER BY date DESC, city
    LIMIT 10
""")

raw_sample


/tmp/ipykernel_34/165350804.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,city,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum
0,Amsterdam,2026-09-25,14.41,20.25,9.60,0.0
1,Bengaluru,2026-09-25,22.73,27.25,21.00,6.7
2,Berlin,2026-09-25,13.06,15.30,10.95,1.2
3,New York,2026-09-25,16.95,21.95,12.35,1.5
4,Sydney,2026-09-25,19.61,23.90,15.60,0.0
5,Amsterdam,2026-09-24,14.14,16.95,11.70,0.1
6,Bengaluru,2026-09-24,23.15,28.95,20.00,1.6
7,Berlin,2026-09-24,13.58,15.55,11.25,2.6
8,New York,2026-09-24,14.69,18.90,10.65,0.0
9,Sydney,2026-09-24,17.83,21.45,13.75,0.1


## 3. Transform with dbt

The dbt layer reads the raw PostgreSQL table through a declared source.

The transformation flow is:

`raw.weather_daily → stg_weather_daily → daily_weather`

The same dbt commands used by the Airflow DAG are executed here.


In [6]:
print("Running dbt models...")

dbt_run_result = dbt("run")

print(f"dbt run return code: {dbt_run_result}")


Running dbt models...
12:37:14  Running with dbt=1.8.8
12:37:14  Registered adapter: postgres=1.8.2
12:37:16  Found 9 data tests, 2 models, 1 source, 423 macros
12:37:16  
12:37:16  Concurrency: 4 threads (target='dev')
12:37:16  
12:37:16  1 of 2 START sql view model public_staging.stg_weather_daily ................... [RUN]
12:37:17  1 of 2 OK created sql view model public_staging.stg_weather_daily .............. [CREATE VIEW in 0.41s]
12:37:17  2 of 2 START sql table model public_marts.daily_weather ........................ [RUN]
12:37:17  2 of 2 OK created sql table model public_marts.daily_weather ................... [SELECT 150 in 0.29s]
12:37:17  
12:37:17  Finished running 1 view model, 1 table model in 0 hours 0 minutes and 1.27 seconds (1.27s).
12:37:17  
12:37:17  Completed successfully
12:37:17  
12:37:17  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 TOTAL=2

dbt run return code: 0


In [7]:
print("Running dbt tests...")

dbt_test_result = dbt("test")

print(f"dbt test return code: {dbt_test_result}")


Running dbt tests...
12:38:12  Running with dbt=1.8.8
12:38:12  Registered adapter: postgres=1.8.2
12:38:13  Found 9 data tests, 2 models, 1 source, 423 macros
12:38:13  
12:38:14  Concurrency: 4 threads (target='dev')
12:38:14  
12:38:14  1 of 9 START test daily_weather_unique ......................................... [RUN]
12:38:14  2 of 9 START test not_null_daily_weather_city .................................. [RUN]
12:38:14  3 of 9 START test not_null_daily_weather_date .................................. [RUN]
12:38:14  4 of 9 START test source_not_null_weather_weather_daily_city ................... [RUN]
12:38:14  1 of 9 PASS daily_weather_unique ............................................... [PASS in 0.26s]
12:38:14  2 of 9 PASS not_null_daily_weather_city ........................................ [PASS in 0.27s]
12:38:14  3 of 9 PASS not_null_daily_weather_date ........................................ [PASS in 0.28s]
12:38:14  4 of 9 PASS source_not_null_weather_weather_daily_c

## 4. Mart Result

The final mart provides a business-recognizable daily weather view for each configured city.

It contains:

- Average temperature
- Maximum temperature
- Minimum temperature
- Total precipitation

The mart is built by dbt as `public_marts.daily_weather`.


In [8]:
mart_result = query("""
    SELECT
        city,
        date,
        avg_temperature,
        max_temperature,
        min_temperature,
        total_precipitation
    FROM public_marts.daily_weather
    ORDER BY date DESC, city
    LIMIT 15
""")

mart_result


/tmp/ipykernel_34/165350804.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,city,date,avg_temperature,max_temperature,min_temperature,total_precipitation
0,Amsterdam,2026-09-25,14.41,20.25,9.60,0.0
1,Bengaluru,2026-09-25,22.73,27.25,21.00,6.7
2,Berlin,2026-09-25,13.06,15.30,10.95,1.2
3,New York,2026-09-25,16.95,21.95,12.35,1.5
4,Sydney,2026-09-25,19.61,23.90,15.60,0.0
5,Amsterdam,2026-09-24,14.14,16.95,11.70,0.1
6,Bengaluru,2026-09-24,23.15,28.95,20.00,1.6
7,Berlin,2026-09-24,13.58,15.55,11.25,2.6
8,New York,2026-09-24,14.69,18.90,10.65,0.0
9,Sydney,2026-09-24,17.83,21.45,13.75,0.1


## 5. Pipeline Summary

The raw layer currently contains 30 days of weather data for 5 configured cities.

This provides 150 city/date records for the assessment period.


In [9]:
summary = query("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT city) AS cities,
        COUNT(DISTINCT date) AS dates,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM raw.weather_daily
""")

summary


/tmp/ipykernel_34/165350804.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,total_rows,cities,dates,min_date,max_date
0,150,5,30,2026-08-27,2026-09-25


## 6. Rerun Safety / Idempotency

To demonstrate rerun safety, the same extraction and load operation is executed again for `2026-09-25`.

Because `(city, date)` is the primary key and the loader uses `ON CONFLICT DO UPDATE`, the rerun updates the existing records instead of creating duplicate rows.


In [10]:
before = query("""
    SELECT COUNT(*) AS row_count
    FROM raw.weather_daily
    WHERE date = '2026-09-25'
""")

print("Rows before rerun:")
display(before)

# Repeat the same extraction and load
rerun_records = fetch_daily("2026-09-25")
load_data(rerun_records)

after = query("""
    SELECT COUNT(*) AS row_count
    FROM raw.weather_daily
    WHERE date = '2026-09-25'
""")

print("Rows after rerun:")
display(after)


Rows before rerun:


/tmp/ipykernel_34/165350804.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,row_count
0,5


Connecting to DB and inserting 5 records...
Load complete!
Rows after rerun:


/tmp/ipykernel_34/165350804.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,row_count
0,5


## 7. Orchestration

The Airflow DAG is named `weather_pipeline` and runs on a daily `@daily` schedule.

The DAG contains four tasks:

1. `extract_weather` — calls the shared `fetch_daily()` function.
2. `load_weather` — retrieves the extracted records through XCom and calls the shared `load_data()` function.
3. `dbt_run` — runs the dbt models.
4. `dbt_test` — runs the dbt data-quality tests.

The dependency chain is:

`extract_weather → load_weather → dbt_run → dbt_test`

The Airflow logical date is passed to the extraction task through the templated `{{ ds }}` value. This means the extraction is driven by the DAG run's logical date rather than the machine's current date, allowing historical dates to be processed during backfills.

No task-level retry policy is explicitly configured in the DAG.


## 8. What I Would Change at Scale

For a larger production workload, I would introduce stronger operational controls around API rate limits, retries, timeouts, logging, alerting, and failure recovery. I would also consider storing immutable raw API responses in object storage before loading them into the warehouse, providing a durable replay point.

For orchestration, I would configure explicit retry policies, task timeouts, concurrency limits, and alerting. Backfills should be controlled so that multiple runs do not unnecessarily compete for the same warehouse resources. The current implementation intentionally keeps the Airflow DAG small for the assessment.

For data quality, I would add freshness checks, accepted-value or range checks for weather measurements, and monitoring for missing city/date combinations. At larger scale, I would also consider incremental dbt models rather than rebuilding the complete mart for every run.
